# Sentiment analysis using Bert and HuggingFace

This code loads a Bert-based Neural Network for sentiment analysis and uses it to evaluate movie reviews.

First we load some required libraries:
  * `torch` for using neural networks
  * `transformers` for downloading the relevant neural networks from Huggingface

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

Now we load the model we want to use for the task - in this case, sentiment analysis. When working with text we need to initialize two pieces of code:
  * the `tokenizer` will receive our input text and turn it into numbers that the neural network can understand.
  * the `model` will receive these numbers (representing our text) and return its best estimate of the sentiment expressed in this text.

In [5]:
# Name of the model we want to use
model_name = "tabularisai/robust-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

We now define an auxiliar function that will hide the complex parts of the neural network.
This function performs the following steps:
  * it stores in the variable `inputs` the number representation of our input text.
  * it feeds the input to the model and stores the response in the `outputs` variable
  * it converts the output of the neural network into probabilities using [the softmax function](https://en.wikipedia.org/wiki/Softmax_function) and stores the result in the `probabilities` variable
  * it identifies which probability is the highest, and uses that to know which classification was selected. The classifications for this model are numbers between 0 and 4, with 0 being the most negative and 4 being the most positive.
  * return a text representation of the above-calculated classification

In [6]:
def predict_sentiment(text):
    inputs = tokenizer(text.lower(), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()

    sentiment_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
    return sentiment_map[predicted_class]


Having defined all of our auxiliary functions we can now focus on our main task: calculate the sentiment of several texts. We define a set of multiple reviews and print their classifications one by one.

In [8]:
texts = ["This movie was less boring than I thought.",
         "I've seen worse.",
         "I loved it, very creative.",
         "Of all the movies I saw this year, this is one of them."
]

for text in texts:
    sentiment = predict_sentiment(text)
    print(f"Text: {text}")
    print(f"Sentiment: {sentiment}\n")

Text: This movie was less boring than I thought.
Sentiment: Negative

Text: I've seen worse.
Sentiment: Very Negative

Text: I loved it, very creative.
Sentiment: Very Positive

Text: Of all the movies I saw this year, this is one of them.
Sentiment: Very Positive

